# 🌪️ โมเดลที่ 1: Typhoon 2.5 LLM Evaluation
**ม.พะเยา | Text Refinement | Profanity Detection | 5W1H Extraction**

## 1. นำเข้าไลบรารีและตั้งค่าระบบ

In [ ]:
import os, sys, json, time, requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['font.family'] = 'TH Sarabun New'
from dotenv import load_dotenv

sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), "..")))
load_dotenv(os.path.join(os.getcwd(), "..", ".env"))

TYPHOON_API_KEY = os.getenv("TYPHOON_API_KEY", "")
TYPHOON_URL     = "https://api.opentyphoon.ai/v1/chat/completions"
MODEL_NAME      = "typhoon-v2.5-30b-a3b-instruct"

print(f"✅ Model: {MODEL_NAME}")
print(f"✅ API Key: {'Configured' if TYPHOON_API_KEY else 'Missing (Demo Mode)'}")
print(f"✅ Endpoint: {TYPHOON_URL}")


## 2. ชุดข้อมูลทดสอบ (Test Dataset)

In [ ]:
TEST_CASES = [
    {
        "id": 1, "type": "Slang → Formal Refinement",
        "raw": "แอร์ห้อง 2304 ตึก PKY มพ. ดับสนิท ร้อนตับแตก ช่วยส่งช่างมาดูด่วนนน",
        "expected_profane": False, "expected_category": "อาคารและสิ่งอำนวยความสะดวก"
    },
    {
        "id": 2, "type": "Profanity & Hate Speech",
        "raw": "ไอ้เวร รถเมล์ มพ. ขับกากชิบหาย เบียดกูเกือบตกข้างทางตรงประตู 1 สัส",
        "expected_profane": True, "expected_category": "การเดินทางและระบบขนส่ง"
    },
    {
        "id": 3, "type": "Cross-Department Event",
        "raw": "เกิดอุบัติเหตุรถเมล์ มพ. ชนสุนัขจรจัดเลือดสาดตรงทางโค้งหน้าตึกสงวนเสริมศรี ขวางทางจราจรมาก",
        "expected_profane": False, "expected_category": "ความปลอดภัยและจราจร"
    },
    {
        "id": 4, "type": "Network Issue Report",
        "raw": "เน็ต UP-WiFi หอพักลุมพินีหลุดบ่อย เชื่อมต่อไม่ได้เลยตั้งแต่หัวค่ำ",
        "expected_profane": False, "expected_category": "ระบบเครือข่ายและเทคโนโลยี"
    },
    {
        "id": 5, "type": "Hygiene & Cleanliness",
        "raw": "ขยะล้นถังขยะหน้าตึก PKY เหม็นมากๆ ไม่มีคนมาเก็บมาหลายวันแล้ว",
        "expected_profane": False, "expected_category": "ภูมิทัศน์และความสะอาด"
    },
]

print(f"✅ Test Dataset: {len(TEST_CASES)} cases")
for tc in TEST_CASES:
    print(f"  #{tc['id']:02d} [{tc['type']}] Expected Profane: {tc['expected_profane']}")


## 3. ฟังก์ชันเรียก Typhoon 2.5 API

In [ ]:
def call_typhoon(prompt: str, temperature: float = 0.2, max_tokens: int = 350) -> str:
    if not TYPHOON_API_KEY:
        return json.dumps({
            "refined_title": "[Mock] ตัวอย่างหัวข้อปัญหาทางการ",
            "refined_detail": "[Mock] ตัวอย่างรายละเอียดภาษาทางการที่สุภาพและครบถ้วน",
            "what": "[Mock] เหตุการณ์ที่เกิดขึ้น",
            "where": "[Mock] สถานที่เกิดเหตุ ม.พะเยา",
            "urgency": "MEDIUM"
        }, ensure_ascii=False)
    payload = {
        "model": MODEL_NAME,
        "messages": [{"role": "user", "content": prompt}],
        "temperature": temperature,
        "max_tokens": max_tokens
    }
    headers = {"Authorization": f"Bearer {TYPHOON_API_KEY}", "Content-Type": "application/json"}
    try:
        res = requests.post(TYPHOON_URL, json=payload, headers=headers, timeout=25)
        if res.status_code == 200:
            return res.json()["choices"][0]["message"]["content"].strip()
        return f"API Error {res.status_code}"
    except Exception as e:
        return f"Exception: {e}"

def check_profanity_api(text: str) -> bool:
    from app.services.ai_service import check_profanity
    return check_profanity(text)

print("✅ API Functions Ready!")


## 4. Running Evaluation — ทดสอบทีละ Case

In [ ]:
PROMPT_5W1H = """
จงสกัดข้อมูล 5W1H และเกลาภาษาข้อความร้องเรียนต่อไปนี้ให้เป็นภาษาทางการสุภาพ:
ข้อความ: "{text}"

ตอบเป็น JSON เท่านั้น:
{{
  "refined_title": "หัวข้อปัญหาภาษาทางการ",
  "refined_detail": "รายละเอียดภาษาทางการสุภาพ",
  "what": "เกิดอะไรขึ้น",
  "where": "สถานที่",
  "urgency": "LOW/MEDIUM/HIGH"
}}
"""

results = []
print("=" * 70)
print("🧪 TYPHOON 2.5 EVALUATION — 5 Test Cases")
print("=" * 70)

for tc in TEST_CASES:
    t0 = time.time()
    is_profane = check_profanity_api(tc["raw"])
    ai_output  = call_typhoon(PROMPT_5W1H.format(text=tc["raw"]))
    latency    = time.time() - t0

    profane_correct = (is_profane == tc["expected_profane"])
    results.append({
        "id": tc["id"], "type": tc["type"],
        "profane_correct": profane_correct,
        "latency": latency,
        "ai_output": ai_output
    })

    status = "✅ PASS" if profane_correct else "❌ FAIL"
    print(f"\n[{status}] Test #{tc['id']}: {tc['type']}")
    print(f"  Input: "{tc['raw'][:60]}..."")
    print(f"  Profanity: {is_profane} (Expected: {tc['expected_profane']})")
    print(f"  Latency: {latency:.2f}s")
    print(f"  Typhoon Output: {ai_output[:120]}...")

passed = sum(1 for r in results if r["profane_correct"])
print(f"\n{'='*70}")
print(f"🏆 Profanity Detection: {passed}/{len(results)} Tests Passed ({passed/len(results)*100:.1f}%)")
print(f"{'='*70}")


## 5. กราฟผลการประเมิน Typhoon 2.5

In [ ]:
# ROUGE-like scores (illustrative based on evaluation doc)
metrics = {
    "Metric": ["ROUGE-1", "ROUGE-2", "ROUGE-L", "Profanity F1", "JSON Validity", "Human Score"],
    "Baseline": [0.52, 0.38, 0.48, 0.82, 0.90, 3.80],
    "Typhoon 2.5": [0.8142, 0.6925, 0.7810, 0.9650, 0.9920, 4.72],
}
df = pd.DataFrame(metrics)

fig, axes = plt.subplots(1, 2, figsize=(15, 6))
fig.suptitle("Typhoon 2.5 — Evaluation Results (ม.พะเยา)", fontsize=14, fontweight='bold')

# Bar Chart: Baseline vs Typhoon
x = np.arange(len(df))
w = 0.35
axes[0].bar(x - w/2, df["Baseline"],    w, label="Baseline", color='#95a5a6', alpha=0.8)
axes[0].bar(x + w/2, df["Typhoon 2.5"], w, label="Typhoon 2.5", color='#3498db', alpha=0.9)
axes[0].set_xticks(x); axes[0].set_xticklabels(df["Metric"], rotation=25, fontsize=9)
axes[0].set_ylim(0, 1.1); axes[0].set_ylabel("Score"); axes[0].legend()
axes[0].set_title("Baseline vs Typhoon 2.5 (All Metrics)", fontweight='bold')
axes[0].axhline(0.90, color='green', linestyle='--', alpha=0.5, label='Target ≥ 0.90')
for i, (b, t) in enumerate(zip(df["Baseline"], df["Typhoon 2.5"])):
    axes[0].text(i - w/2, b + 0.02, f"{b:.2f}", ha='center', fontsize=8)
    axes[0].text(i + w/2, t + 0.02, f"{t:.2f}", ha='center', fontsize=8, fontweight='bold', color='#2980b9')

# Latency pie chart
latencies = [r["latency"] for r in results]
labels_lat = [f"#{r['id']}" for r in results]
axes[1].pie(latencies, labels=labels_lat, autopct='%1.1f%%', startangle=140,
            colors=['#3498db', '#e74c3c', '#2ecc71', '#f39c12', '#9b59b6'])
axes[1].set_title(f"API Latency Distribution\n(Total: {sum(latencies):.1f}s for {len(results)} calls)", fontweight='bold')

plt.tight_layout()
plt.savefig("typhoon_evaluation.png", dpi=150, bbox_inches='tight')
plt.show()

print(f"\nAverage API Latency: {np.mean(latencies):.2f}s per call")
